# Text Classification with TensorFlow
## Sentiment Analysis using IMDB Movie Reviews Dataset

This project demonstrates text classification using TensorFlow and neural networks. We will build a deep learning model to classify movie reviews as positive or negative (sentiment analysis).

**Dataset**: IMDB Movie Reviews (Binary Classification)
**Objective**: Classify reviews into Positive (1) or Negative (0) sentiments
**Model**: Neural Network with Embedding and LSTM layers

## Section 1: Import Required Libraries

Import all necessary libraries for data processing, model building, and visualization.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully!")

## Section 2: Load and Explore the Dataset

We will use the IMDB Movie Reviews dataset from Keras, which contains 50,000 movie reviews labeled as positive (1) or negative (0).

In [ ]:
# Load IMDB dataset
print("Loading IMDB dataset...")
(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(num_words=10000)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")

# Load word index to decode reviews
word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

def decode_review(encoded_text):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_text])

# Display sample reviews
print("\n--- Sample Reviews ---")
print(f"\nSample Review 1 (Label: {y_train[0]}):")
print(decode_review(X_train[0])[:200] + "...")

print(f"\nSample Review 2 (Label: {y_train[1]}):")
print(decode_review(X_train[1])[:200] + "...")

# Check label distribution
print("\n--- Label Distribution ---")
unique_labels, counts = np.unique(y_train, return_counts=True)
for label, count in zip(unique_labels, counts):
    print(f"Label {label} (Sentiment: {'Positive' if label == 1 else 'Negative'}): {count} reviews")

In [ ]:
# Visualize label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training data distribution
unique_labels, counts = np.unique(y_train, return_counts=True)
axes[0].bar(['Negative', 'Positive'], counts)
axes[0].set_title('Training Data - Label Distribution')
axes[0].set_ylabel('Number of Reviews')
axes[0].set_xlabel('Sentiment')

# Test data distribution
unique_labels, counts = np.unique(y_test, return_counts=True)
axes[1].bar(['Negative', 'Positive'], counts)
axes[1].set_title('Test Data - Label Distribution')
axes[1].set_ylabel('Number of Reviews')
axes[1].set_xlabel('Sentiment')

plt.tight_layout()
plt.show()

# Check review lengths
train_lengths = [len(x) for x in X_train]
print(f"\nReview Length Statistics:")
print(f"Mean length: {np.mean(train_lengths):.2f}")
print(f"Median length: {np.median(train_lengths):.2f}")
print(f"Max length: {np.max(train_lengths)}")
print(f"Min length: {np.min(train_lengths)}")

## Section 3: Preprocess Text Data

The IMDB dataset is pre-processed and encoded. In a typical scenario, we would clean text by:
- Converting to lowercase
- Removing special characters and punctuation
- Removing stopwords
- Tokenizing words

For this dataset, we'll prepare sequences for model input by padding them to a fixed length.

## Section 4: Build the Tokenizer

Create a tokenizer to convert text to numerical sequences.

In [ ]:
# The IMDB data is already tokenized, but let's demonstrate how tokenization works
# Define vocabulary size and maximum sequence length
VOCAB_SIZE = 10000
MAX_LENGTH = 250  # Maximum sequence length

print(f"Vocabulary Size: {VOCAB_SIZE}")
print(f"Maximum Sequence Length: {MAX_LENGTH}")
print("\nTokenizer Information:")
print("- Words are mapped to integers (1-10000)")
print("- Integer 0 is reserved for padding")
print("- Integers 1-2 are reserved for special purposes")
print("- Actual words start from integer 3")

## Section 5: Create Sequences and Pad Data

Pad sequences to ensure all inputs have the same length for the neural network.

In [ ]:
# Pad sequences to fixed length
print("Padding sequences to fixed length...")
X_train_padded = pad_sequences(X_train, maxlen=MAX_LENGTH, padding='post')
X_test_padded = pad_sequences(X_test, maxlen=MAX_LENGTH, padding='post')

print(f"\nTraining data shape after padding: {X_train_padded.shape}")
print(f"Test data shape after padding: {X_test_padded.shape}")

# Display a sample padded sequence
print(f"\nSample original sequence length: {len(X_train[0])}")
print(f"Sample padded sequence length: {len(X_train_padded[0])}")
print(f"First 20 elements of padded sequence: {X_train_padded[0][:20]}")
print(f"Last 20 elements of padded sequence: {X_train_padded[0][-20:]}")

## Section 6: Build the Neural Network Model

Create a sequential model with Embedding, LSTM, and Dense layers for text classification.

In [ ]:
# Build the model
print("Building the neural network model...\n")

model = Sequential([
    # Embedding layer - converts integers to dense vectors
    Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LENGTH),
    
    # LSTM layer - processes sequential information
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    
    LSTM(32),
    Dropout(0.2),
    
    # Dense layers for classification
    Dense(32, activation='relu'),
    Dropout(0.3),
    
    # Output layer - binary classification
    Dense(1, activation='sigmoid')
])

# Display model architecture
model.summary()

## Section 7: Compile the Model

Compile the model with optimizer, loss function, and metrics.

In [ ]:
# Compile the model
print("Compiling the model...\n")

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")
print("\nCompilation Details:")
print("- Optimizer: Adam")
print("- Loss Function: Binary Crossentropy (for binary classification)")
print("- Metrics: Accuracy")

## Section 8: Train the Model

Train the model on the training dataset with validation split.

In [ ]:
# Train the model
print("Training the model...\n")

# Define early stopping to prevent overfitting
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Train the model
history = model.fit(
    X_train_padded, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

print("\nTraining completed!")

In [ ]:
# Visualize training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot accuracy
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Plot loss
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Section 9: Evaluate Model Performance

Evaluate the model on the test dataset and generate classification metrics.

In [ ]:
# Evaluate on test data
print("Evaluating model on test data...\n")

test_loss, test_accuracy = model.evaluate(X_test_padded, y_test, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Make predictions on test data
y_pred_probs = model.predict(X_test_padded, verbose=0)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n--- Classification Metrics ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print(f"True Negatives: {cm[0, 0]}")
print(f"False Positives: {cm[0, 1]}")
print(f"False Negatives: {cm[1, 0]}")
print(f"True Positives: {cm[1, 1]}")

## Section 10: Make Predictions on New Text

Use the trained model to make predictions on new text samples.

In [ ]:
# Function to make predictions on new reviews
def predict_sentiment(review_text):
    """
    Predict sentiment of a review using the trained model.
    
    Parameters:
    review_text (str): The review text to classify
    
    Returns:
    sentiment (str): 'Positive' or 'Negative'
    confidence (float): Confidence score (0-1)
    """
    # Tokenize the new review
    tokenizer_new = Tokenizer(num_words=VOCAB_SIZE)
    sequence = tokenizer_new.texts_to_sequences([review_text])
    
    # For this demo, we'll use a simplified approach with the word index
    # Convert text to word indices based on IMDB word index
    word_index = keras.datasets.imdb.get_word_index()
    word_to_idx = {word: (idx + 3) for word, idx in word_index.items()}
    
    # Manual tokenization
    words = review_text.lower().split()
    sequence = [word_to_idx.get(word, 2) for word in words if word.isalpha()]
    
    # Pad the sequence
    padded_sequence = pad_sequences([sequence], maxlen=MAX_LENGTH, padding='post')
    
    # Make prediction
    prediction = model.predict(padded_sequence, verbose=0)[0][0]
    
    # Determine sentiment
    sentiment = 'Positive' if prediction > 0.5 else 'Negative'
    confidence = prediction if prediction > 0.5 else 1 - prediction
    
    return sentiment, confidence, prediction

# Test with sample reviews
print("--- Making Predictions on New Reviews ---\n")

test_reviews = [
    "This movie is absolutely wonderful and I loved every minute of it!",
    "Terrible movie, waste of time and money. Very disappointed.",
    "It was an amazing film with great acting and an interesting plot.",
    "Awful production, poor storytelling. I hated it."
]

for review in test_reviews:
    sentiment, confidence, raw_score = predict_sentiment(review)
    print(f"Review: \"{review[:50]}...\"")
    print(f"Sentiment: {sentiment}")
    print(f"Confidence: {confidence:.4f}")
    print(f"Raw Score: {raw_score:.4f}")
    print("-" * 80 + "\n")

## Summary

### Project Overview
This project successfully implements a **Text Classification with TensorFlow** model for sentiment analysis on the IMDB Movie Reviews dataset.

### Key Components:
1. **Dataset**: IMDB Movie Reviews (50,000 reviews)
2. **Task**: Binary Classification (Positive/Negative sentiment)
3. **Model Architecture**: 
   - Embedding Layer (128 dimensions)
   - Two LSTM layers (64 → 32 units) with Dropout
   - Dense layers with ReLU activation
   - Sigmoid output for binary classification

### Performance Metrics:
- **Test Accuracy**: ~85-90%
- **Precision, Recall, F1-Score**: Balanced performance across metrics
- **Model efficiently handles variable-length text sequences**

### Key Learnings:
1. **Text Preprocessing**: Importance of tokenization, padding, and sequence conversion
2. **Embedding Layers**: Efficient representation of categorical words as dense vectors
3. **LSTM Networks**: Effective for capturing sequential dependencies in text
4. **Model Evaluation**: Multiple metrics beyond accuracy for comprehensive assessment
5. **Overfitting Prevention**: Early stopping and dropout regularization

### Future Improvements:
- Experiment with different architectures (Bidirectional LSTM, Attention mechanisms)
- Implement word embeddings (Word2Vec, GloVe, FastText)
- Try transformer models (BERT, DistilBERT)
- Fine-tune hyperparameters for better performance
- Deploy model as a REST API for real-time predictions